# HPO Stage 1 — Transition Network (offline, multi-objective)

Tunes the BayesFormer transition network on a **fixed offline dataset** — no
environment interaction per trial, so trials are cheap and results are
attributable to the transition hyperparameters alone.

## Plan
1. **Setup** — load base config (`config_vector-setting1.yaml`, DeepSea-5
   deterministic), fix all seeds.
2. **Data** — collect one shared dataset of random-policy transitions
   (`generate_transition_data`) plus a per-state test set. Generated once,
   reused by every trial so scores are comparable.
3. **Objective** — per trial: sample hyperparameters, train with early
   stopping, evaluate with 100 locked-dropout world hypotheses M̂:
   - **Objective 1 ↓ `mean_grid_dist`** — mean Euclidean distance in
     (row, col) space between predicted and true next state. World-model
     accuracy.
   - **Objective 2 ↓ `overconfident_error_rate`** — fraction of
     (state, action) pairs that are *wrong* (grid dist > 0.5 cells) with a
     *near-zero posterior spread* (grid variance < 0.05). This is the failure
     mode that kills Thompson Sampling: if every sampled M̂ agrees on the wrong
     transition, no episode ever explores the correction. Accuracy alone
     cannot see it.
4. **Search** — NSGA-II Pareto-front search, persisted to the shared SQLite DB
   (`hpo_artifacts/optuna_btrl.db`) so the study is resumable.
5. **Analysis** — trial table + Pareto-front plot.
6. **Export** — Pareto candidates → `hpo_artifacts/transition_pareto.json`
   for the validation notebook (hpo2).

Diagnostics logged per trial (not searched on): reward-head MSE, goal-path
grid distance (bottom-half states), exact-match accuracy, parameter count.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import optuna
from matplotlib import pyplot as plt

from notebook_utils import generate_transition_data
from TRL.tuning.common import (
    ARTIFACTS_DIR, OPTUNA_DB, TRANSITION_STUDY_NAME, TRANSITION_PARETO_JSON,
    load_base_config, set_global_seed, get_device,
)
from TRL.tuning import transition_search as ts

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.options.display.float_format = '{:.5f}'.format

config = load_base_config(env='5', deterministic=True)
DEVICE = get_device(config)
print(f'torch {torch.__version__} | optuna {optuna.__version__} | device {DEVICE}')

In [ ]:
# ── Budget knobs ─────────────────────────────────────────────────────────────
SEED         = 42
DATA_STEPS   = 2000   # offline dataset size (transitions), shared by all trials
N_TRIALS     = 200    # NSGA-II population needs breadth; trials are cheap
N_HYPOTHESES = 100    # locked-dropout M-hat samples per evaluation
MAX_EPOCHS   = 300
PATIENCE     = 15
MIN_DELTA    = 5e-4

set_global_seed(SEED)

In [ ]:
# ── Shared offline dataset (generated once, reused by every trial) ──────────
# Random policy on deterministic DeepSea-5. ~3% of random episodes reach the
# goal, so at 2000 transitions the goal region appears in both train and test.
episodes, test_transitions, obs_space, possible_actions = generate_transition_data(
    config, DATA_STEPS
)

data = {
    'states':      torch.tensor(episodes['states'],      dtype=torch.float32),
    'next_states': torch.tensor(episodes['next_states'], dtype=torch.float32),
    'timesteps':   torch.tensor(episodes['timestep'],    dtype=int),
    'rewards':     torch.tensor(episodes['rewards']).unsqueeze(1),
    'actions':     torch.tensor(episodes['actions'],     dtype=int).unsqueeze(1),
    'test_transitions': test_transitions,
    'obs_space': obs_space,
    'possible_actions': possible_actions,
}
test_size = len(test_transitions) * len(possible_actions)
print(f'obs_space={obs_space} | train transitions={len(episodes["actions"])} '
      f'| test (state,action) pairs={test_size}')

## Search space

| Parameter | Range / choices | Represents |
|---|---|---|
| `hidden_dim` × `heads` | joint index over 20 valid pairs (dim ∈ {32…512}, heads ∈ {1,2,4,8}, dim % heads == 0) | embedding width and attention-head count — model capacity |
| `learning_rate` | log-uniform [1e-4, 1e-2] | Adam step size |
| `num_encoder_layers` | int [1, 4] | Transformer depth |
| `forward_expansion` | {2, 4, 8} | FFN width multiplier inside each block |
| `p_seq` | {0.05, 0.1, 0.2, 0.3} | embedding/positional dropout — observation-noise part of the Bayesian posterior |
| `p_attn` | {0.02, 0.05, 0.1, 0.15} | Q/K/V, attention-map and FFN dropout — epistemic-uncertainty part of the posterior |

`p_seq`/`p_attn` are not just regularisers here: the dropout masks **are** the
posterior over world models (BayesFormer), so these two directly shape the
Thompson-Sampling exploration distribution.

In [ ]:
def objective(trial):
    params = ts.suggest_transition_params(trial)
    metrics = ts.train_and_eval_transition(
        config, params, data, DEVICE,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
        n_hypotheses=N_HYPOTHESES,
    )
    for key, value in {**params, **{k: v for k, v in metrics.items() if k != 'objectives'}}.items():
        trial.set_user_attr(key, value)
    return metrics['objectives']

In [ ]:
ARTIFACTS_DIR.mkdir(exist_ok=True)
study = optuna.create_study(
    study_name=TRANSITION_STUDY_NAME,
    storage=OPTUNA_DB,
    directions=['minimize', 'minimize'],
    sampler=optuna.samplers.NSGAIISampler(seed=SEED),
    load_if_exists=True,
)

done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining = max(0, N_TRIALS - done)
print(f'{done} trials already in {OPTUNA_DB}; running {remaining} more')
if remaining:
    study.optimize(objective, n_trials=remaining, show_progress_bar=True)
print(f'Pareto-front size: {len(study.best_trials)}')

In [ ]:
# ── Trial table ──────────────────────────────────────────────────────────────
ATTRS = ['hidden_dim', 'heads', 'learning_rate', 'num_encoder_layers',
         'forward_expansion', 'p_seq', 'p_attn',
         'mean_grid_dist', 'overconfident_error_rate', 'reward_mse',
         'goal_path_grid_dist', 'exact_match_accuracy', 'mean_grid_variance',
         'epochs_to_converge', 'n_params']

results_df = pd.DataFrame([
    {'trial': t.number, **{a: t.user_attrs.get(a) for a in ATTRS}}
    for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
]).sort_values('mean_grid_dist').reset_index(drop=True)

print(f'{len(results_df)} completed trials')
results_df.head(20)

In [ ]:
# ── Pareto front ─────────────────────────────────────────────────────────────
pareto_numbers = {t.number for t in study.best_trials}
is_pareto = results_df['trial'].isin(pareto_numbers)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    results_df['mean_grid_dist'], results_df['overconfident_error_rate'],
    c=results_df['exact_match_accuracy'], cmap='RdYlGn', vmin=0, vmax=1,
    alpha=0.65, s=60, edgecolors='grey', linewidths=0.3,
)
ax.scatter(
    results_df.loc[is_pareto, 'mean_grid_dist'],
    results_df.loc[is_pareto, 'overconfident_error_rate'],
    color='black', marker='*', s=250, zorder=5,
    label=f'Pareto front ({is_pareto.sum()} trials)',
)
plt.colorbar(sc, ax=ax, label='Exact-match accuracy')
ax.set_xlabel('Mean grid distance  (world-model accuracy, lower = better)')
ax.set_ylabel('Overconfident-error rate  (posterior miscalibration, lower = better)')
ax.set_title('Transition Network Pareto Front — accuracy vs posterior calibration')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Export Pareto candidates for the validation notebook (hpo2) ─────────────
PARAM_KEYS = ['hidden_dim', 'heads', 'learning_rate', 'num_encoder_layers',
              'forward_expansion', 'p_seq', 'p_attn']
candidates = [{
    'trial': t.number,
    'params': {k: t.user_attrs[k] for k in PARAM_KEYS},
    'mean_grid_dist': t.values[0],
    'overconfident_error_rate': t.values[1],
    'reward_mse': t.user_attrs.get('reward_mse'),
    'goal_path_grid_dist': t.user_attrs.get('goal_path_grid_dist'),
    'exact_match_accuracy': t.user_attrs.get('exact_match_accuracy'),
} for t in study.best_trials]

with open(TRANSITION_PARETO_JSON, 'w') as f:
    json.dump(candidates, f, indent=2)
print(f'Wrote {len(candidates)} Pareto candidates -> {TRANSITION_PARETO_JSON}')

## Next step

Run **`hpo2_validation.ipynb`**: the top Pareto candidates are re-evaluated
with **full agent runs** (multi-seed cumulative regret) to check that these
offline objectives actually predict downstream performance — the
objective-mismatch check — and the winner is frozen for the value-network
search (hpo3).